# snRNA-seq processing after sample merging

This notebook performs sample loading/merging, doublet detection, QC, normalization, dimensionality reduction, Harmony integration, Leiden clustering, manual cell-type annotation, and summary visualization.

## Reproducibility notes
- Raw data and large `.h5ad` intermediates are **not** intended for GitHub.
- Set `SNRNASEQ_METADATA` to the semicolon-delimited sample metadata file, or place `All_withall4plex.csv` in `SNRNASEQ_DATA_DIR`.
- The metadata table is expected to contain `Path_to_outs`, `Sample_ID`, `Sample`, `Library`, `Sample_name`, `Batch_nuclei`, `Sample_type`, and `snRNAseq_type`.
- QC and doublet thresholds below are study-specific analysis choices and are kept explicit for traceability.
- Notebook outputs were cleared before publication to keep the repository small and avoid exposing local paths/data.

In [ ]:
# Standard library
import logging
import os
from pathlib import Path

# Scientific Python / scverse
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

# Plot/export configuration
mpl.rcParams["pdf.fonttype"] = 42
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"] = 42
# Run `logging.getLogger('fontTools').setLevel` for this analysis step.
logging.getLogger("fontTools").setLevel(logging.WARNING)
# Run `logging.getLogger('fontTools.subset').setLevel` for this analysis step.
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

# Reproducible paths. Override SNRNASEQ_DATA_DIR / SNRNASEQ_METADATA as needed.
DATA_DIR = Path(os.environ.get("SNRNASEQ_DATA_DIR", ".")).resolve()
# Compute and store `METADATA_PATH`.
METADATA_PATH = Path(os.environ.get("SNRNASEQ_METADATA", DATA_DIR / "All_withall4plex.csv"))
# Compute and store `OUTPUT_DIR`.
OUTPUT_DIR = Path(os.environ.get("SNRNASEQ_OUTPUT_DIR", "results"))
# Compute and store `FIGURE_DIR`.
FIGURE_DIR = Path(os.environ.get("SNRNASEQ_FIGURE_DIR", "figures"))
# Run `OUTPUT_DIR.mkdir` for this analysis step.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Run `FIGURE_DIR.mkdir` for this analysis step.
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Run `print` for this analysis step.
print(f"Scanpy {sc.__version__}; AnnData {ad.__version__}")
# Run `print` for this analysis step.
print(f"Metadata: {METADATA_PATH}")


## 1. Load metadata and merge samples

In [ ]:
# Load the supporting metadata table.
metadata_df = pd.read_csv(METADATA_PATH, sep=";")


In [ ]:
# Inspect the first rows of the resulting table.
metadata_df.head()


In [ ]:
# Run `metadata_df['Sample_ID'].unique` for this analysis step.
metadata_df["Sample_ID"].unique()


In [ ]:
# List to hold all AnnData objects
adata_list = []

# Loop through the metadata and load each sample
for idx, row in metadata_df.iterrows():
    matrix_path = row['Path_to_outs']
    
    # Check if the path exists
    if Path(matrix_path).exists():
        # Load the .h5 file
        adata = sc.read_10x_h5(matrix_path)
        
        # Modify the cell names to include the sample_id
        adata.obs_names = [f"{row['Sample_ID']}_{cell_name}" for cell_name in adata.obs_names]
        
        # Add the sample-specific metadata to the AnnData object
        adata.obs['id'] = row['Sample_ID']
        adata.obs['sample'] = row['Sample']
        adata.obs['library'] = row['Library']
        adata.obs['name'] = row['Sample_name']
        adata.obs['batch_nuclei'] = row['Batch_nuclei']
        adata.obs['sample_type'] = row['Sample_type']
        adata.obs['plex'] = row['snRNAseq_type']
        
        # Make var_names unique if they are not already
        adata.var_names_make_unique()
        
        # Append to list
        adata_list.append(adata)
    else:
        print(f"Matrix file for sample {row['Sample_ID']} not found at {matrix_path}")
        
# Concatenate all AnnData objects using ad.concat
adata = ad.concat(adata_list, index_unique='-')

# Verify the result
print(adata)

In [ ]:
# Check the first few entries of .obs (metadata)
print(adata.obs.head())

In [ ]:
adata.shape

In [ ]:
# Save the processed AnnData object to disk.
adata.write_h5ad(OUTPUT_DIR / "Allwithall4plex.h5ad")


In [ ]:
# Load the processed AnnData object.
adata = sc.read_h5ad(OUTPUT_DIR / "Allwithall4plex.h5ad")


In [ ]:
# Inspect the number of observations in each category.
adata.obs['name'].value_counts()


In [ ]:
# Set `out_dir` for the following analysis.
out_dir = FIGURE_DIR


## 2. Pre-filtering summaries

In [ ]:
# Reset the DataFrame index after reshaping or filtering.
name_counts = (
    adata.obs["name"]
    .value_counts(sort=False)
    .rename_axis("name")
    .reset_index(name="cell_count")
)

# Export the resulting table as a CSV file.
name_counts.to_csv(out_dir / "cell_counts_per_sample_beforefiltering_snRNAseq.csv", index=False)


In [ ]:
name_counts

In [ ]:
# Check the first few entries of .obs (metadata)
print(adata.obs.head())

In [ ]:
# Group by sample_id and get unique sample_names for each sample_id
unique_names_per_id = adata.obs.groupby('id')['name'].unique()

# Print the summary
print(unique_names_per_id)

In [ ]:
adata.shape

## 3. Doublet detection with Scrublet

In [ ]:
#Want to run scrublet on individual samples 
# Create empty containers
doublet_scores_all = []
# Define the values used for `predicted_doublets_all`.
predicted_doublets_all = []

# Repeat the following operation for each item in the selected collection.
for sample in adata.obs['id'].unique():
    print(f"Running Scanpy Scrublet on sample: {sample}")
    
    # Subset to this sample
    adata_id = adata[adata.obs['id'] == sample].copy()
    
    # Run scrublet (Scanpy wrapper)
    sc.pp.scrublet(adata_id)
    
    # Append results (keep same order as main adata)
    doublet_scores_all.append(pd.Series(adata_id.obs['doublet_score'], index=adata_id.obs_names))
    predicted_doublets_all.append(pd.Series(adata_id.obs['predicted_doublet'], index=adata_id.obs_names))

# Concatenate per-cell results
doublet_scores_all = pd.concat(doublet_scores_all)
# Combine the selected objects into one dataset.
predicted_doublets_all = pd.concat(predicted_doublets_all)

# Add to main adata
adata.obs['doublet_score'] = doublet_scores_all
# Set `adata.obs['predicted_doublet']` for the following analysis.
adata.obs['predicted_doublet'] = predicted_doublets_all

# Verify
adata.obs[['id', 'doublet_score', 'predicted_doublet']].head()


In [ ]:
# Save the processed AnnData object to disk.
adata.write_h5ad(OUTPUT_DIR / "Allwithall4plex_postscrublet.h5ad")


## 4. Quality-control metrics

In [ ]:
# Flag mitochondrial genes (mouse naming convention).
adata.var["mt"] = adata.var_names.str.startswith("mt-")

In [ ]:
#If want to check if a specific gene name is present - wanted to check if indeed the genes are in small letters, bc the scanpy tutorial di dnot specify it for mouse
#Here g is the same as i in R basically 
[g for g in adata.var_names if "RPS" in g or "Rps" in g][:20]

In [ ]:
# Flag ribosomal and hemoglobin genes for QC summaries.
adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))
# Compute and store `adata.var['hb']`.
adata.var["hb"] = adata.var_names.str.contains(r"^Hb(?!p)", regex=True)


In [ ]:
#Sanity check - see how many genes got flagged
adata.var["mt"].sum()

In [ ]:
#calculates common quality control (QC) metrics
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

In [ ]:
#Visualise - n_genes_by_counts are basically features per cell - genes per cell that have more than one count per cell
#total_counts - are UMIs per cell
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="id",
    multi_panel=True,
)

In [ ]:
#Visualise - n_genes_by_counts are basically features per cell - genes per cell that have more than one count per cell
#total_counts - are UMIs per cell
sc.pl.violin(
    adata,
    ["n_genes_by_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="id"
)

In [ ]:
#Visualise - n_genes_by_counts are basically features per cell - genes per cell that have more than one count per cell
#total_counts - are UMIs per cell
sc.pl.violin(
    adata,
    ["n_genes_by_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="sample"
)

In [ ]:
#Visualise - n_genes_by_counts are basically features per cell - genes per cell that have more than one count per cell
#total_counts - are UMIs per cell
sc.pl.violin(
    adata,
    ["n_genes_by_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="plex"
)

In [ ]:
#Visualise - n_genes_by_counts are basically features per cell - genes per cell that have more than one count per cell
sc.pl.violin(
    adata,
    ["pct_counts_mt"],
    jitter=0, #default was 0.4 but gets too dense, cant see anything
    groupby="id"
)

In [ ]:
# Run `sc.pl.scatter` for this analysis step.
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")


### Inspect doublet-score distributions

In [ ]:
# Compute and store `samples`.
samples = adata.obs['id'].unique()
# Compute and store `n`.
n = len(samples)
# Set `cols` for the following analysis.
cols = 3
# Compute and store `rows`.
rows = int(np.ceil(n / cols))

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(4*cols, 3*rows))
# Repeat the following operation for each item in the selected collection.
for i, s in enumerate(samples, 1):
    ax = plt.subplot(rows, cols, i)
    x = adata.obs.loc[adata.obs['id']==s, 'doublet_score'].values
    ax.hist(x, bins=50)
    ax.set_title(s)
    ax.set_xlabel('doublet_score')
    ax.set_ylabel('cells')
# Run `plt.tight_layout` for this analysis step.
plt.tight_layout()
# Run `plt.show` for this analysis step.
plt.show()


### Apply sample-specific doublet thresholds

In [ ]:
# Sample-specific Scrublet score thresholds chosen after inspecting score distributions.
DOUBLET_SCORE_CUTOFFS = {
    "AKH34_Lib1": 0.11,
    "AKH38_2_Lib1": 0.15,
    "AKH49_Lib1": 0.10,
    "AKH34_Lib2": 0.15,
    "AKH34_2_Lib2": 0.15,
    "AKH38_Lib2": 0.12,
    "AKH50": 0.25,
}
# Set `DEFAULT_DOUBLET_SCORE_CUTOFF` for the following analysis.
DEFAULT_DOUBLET_SCORE_CUTOFF = 0.19

# Compute and store `thresholds`.
thresholds = adata.obs["id"].map(DOUBLET_SCORE_CUTOFFS).fillna(DEFAULT_DOUBLET_SCORE_CUTOFF)
# Set `adata.obs['predicted_doublet_custom']` for the following analysis.
adata.obs["predicted_doublet_custom"] = adata.obs["doublet_score"] > thresholds


## 5. Cell and gene filtering

In [ ]:
# Cell-level QC thresholds. These are analysis decisions and should be reported with the study.
MIN_GENES = 200
# Set `MAX_GENES` for the following analysis.
MAX_GENES = 4_000
# Set `MIN_COUNTS` for the following analysis.
MIN_COUNTS = 300
# Set `MAX_COUNTS` for the following analysis.
MAX_COUNTS = 12_000
# Set `MAX_PCT_MT` for the following analysis.
MAX_PCT_MT = 0.5

# Make an independent copy of the selected data.
adata_unfiltered = adata.copy()
# Calculate `qc_mask` from the existing values.
qc_mask = (
    adata.obs["n_genes_by_counts"].between(MIN_GENES, MAX_GENES)
    & adata.obs["total_counts"].between(MIN_COUNTS, MAX_COUNTS)
    & (adata.obs["pct_counts_mt"] < MAX_PCT_MT)
)
# Make an independent copy of the selected data.
adata_qc = adata[qc_mask].copy()


In [ ]:
# Filter genes using the specified detection threshold.
sc.pp.filter_genes(adata_qc, min_cells=3)


In [ ]:
#Check number of cells
adata_qc.obs["id"].value_counts()

In [ ]:
# Inspect the number of observations in each category.
adata_qc.obs["predicted_doublet_custom"].value_counts()


In [ ]:
#remove predicted doublets
adata_qc = adata_qc[~adata_qc.obs["predicted_doublet_custom"]].copy()

In [ ]:
#Check number of cells
adata_qc.obs["id"].value_counts()

In [ ]:
#Visualise
sc.pl.violin(
    adata_qc,
    ["n_genes_by_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="id"
)

In [ ]:
#Visualise
sc.pl.violin(
    adata_qc,
    ["n_genes_by_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="sample"
)

In [ ]:
#Visualise
sc.pl.violin(
    adata_qc,
    ["n_genes_by_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="plex"
)

In [ ]:
#Visualise
sc.pl.violin(
    adata_qc,
    ["total_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="id"
)

In [ ]:
#Visualise
sc.pl.violin(
    adata_qc,
    ["total_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="sample"
)

In [ ]:
#Visualise
sc.pl.violin(
    adata_qc,
    ["total_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="plex"
)

In [ ]:
#Check number of cells
adata_qc.obs["id"].value_counts()
adata_qc.shape

In [ ]:
# Save the processed AnnData object to disk.
adata_qc.write_h5ad(OUTPUT_DIR / "filtered_019nodoublets_all_withall4plex.h5ad")


In [ ]:
# Load the processed AnnData object.
adata_qc = sc.read_h5ad(OUTPUT_DIR / "filtered_019nodoublets_all_withall4plex.h5ad")


## 6. Post-filtering summaries

In [ ]:
# Reset the DataFrame index after reshaping or filtering.
name_counts = (
    adata_qc.obs["name"]
    .value_counts(sort=False)
    .rename_axis("name")
    .reset_index(name="cell_count")
)

# Export the resulting table as a CSV file.
name_counts.to_csv(out_dir / "cell_counts_per_sample_afterfiltering_snRNAseq.csv", index=False)


In [ ]:
name_counts

## 7. Normalization and highly variable genes

In [ ]:
# Saving count data
adata_qc.layers["counts"] = adata_qc.X.copy()

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata_qc)
# Logarithmize the data
sc.pp.log1p(adata_qc)

In [ ]:
# Identify highly variable genes.
sc.pp.highly_variable_genes(adata_qc, n_top_genes=2000, batch_key="id")


In [ ]:
# Run `sc.pl.highly_variable_genes` for this analysis step.
sc.pl.highly_variable_genes(adata_qc)


In [ ]:
# Exclude experimental construct genes from the HVG set.
genes_to_exclude = ["C_terminal_Cas9", "Abe8e_with_N_terminal_Cas9"]
# Set `adata_qc.var.loc[adata_qc.var_names.isin(genes_to_exclude), 'highly_variable']` for the following analysis.
adata_qc.var.loc[adata_qc.var_names.isin(genes_to_exclude), "highly_variable"] = False


## 8. Scaling, PCA, neighbors, and UMAP

In [ ]:
# Keep a separate layer for scaling so log-normalized expression remains in `adata_qc.X`.
adata_qc.layers["scaled"] = adata_qc.X.copy()

In [ ]:
# Scale gene expression for downstream dimensionality reduction.
sc.pp.scale(adata_qc, max_value=10, layer="scaled")


In [ ]:
# Compute principal components.
sc.tl.pca(adata_qc, use_highly_variable=True, layer = "scaled")


In [ ]:
# Run `sc.pl.pca_variance_ratio` for this analysis step.
sc.pl.pca_variance_ratio(adata_qc, n_pcs=50, log=True)


In [ ]:
# Run `sc.pl.pca_loadings` for this analysis step.
sc.pl.pca_loadings(adata_qc, components=(1,2,3,4), include_lowest=True)


In [ ]:
# Run `sc.pl.highest_expr_genes` for this analysis step.
sc.pl.highest_expr_genes(adata_qc, n_top=20)


In [ ]:
#n_neighbors=15 is default
sc.pp.neighbors(adata_qc, n_pcs=50)

In [ ]:
# Compute the UMAP embedding.
sc.tl.umap(adata_qc)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(
    adata_qc,
    color="id",
    # Setting a smaller point size to get prevent overlap
    size=3
)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(
    adata_qc,
    color="name",
    # Setting a smaller point size to get prevent overlap
    size=3
)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(
    adata_qc,
    color="sample_type",
    # Setting a smaller point size to get prevent overlap
    size=3
)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(
    adata_qc,
    color="plex",
    # Setting a smaller point size to get prevent overlap
    size=3
)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(
    adata_qc,
    color="doublet_score",
    # Setting a smaller point size to get prevent overlap
    size=3
)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc[adata_qc.obs["sample_type"] == "BE"], color="id", size = 5)


In [ ]:
# Compute and store `adata_qc.obs['highlight']`.
adata_qc.obs["highlight"] = np.where(
    adata_qc.obs["sample_type"] == "BE",
    adata_qc.obs["id"],
    "Other"
)
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc, color="highlight", palette=["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd", "lightgrey"], size=3)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc[adata_qc.obs["sample_type"] == "R636Q"], color="id", size = 3)


In [ ]:
# Compute and store `adata_qc.obs['highlight']`.
adata_qc.obs["highlight"] = np.where(
    adata_qc.obs["sample_type"] == "R636Q",
    adata_qc.obs["id"],
    "Other"
)
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc, color="highlight", palette=["#1f77b4", "#ff7f0e", "#2ca02c", "lightgrey"], size=3)


In [ ]:
# Compute and store `adata_qc.obs['highlight']`.
adata_qc.obs["highlight"] = np.where(
    adata_qc.obs["sample_type"] == "WT",
    adata_qc.obs["id"],
    "Other"
)
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc, color="highlight", palette=["#1f77b4", "#ff7f0e", "lightgrey"], size=3)


In [ ]:
# Save the processed AnnData object to disk.
adata_qc.write_h5ad(OUTPUT_DIR / "filtered_scaled_umap_nodoublets_all_withall4plex.h5ad")


In [ ]:
# Load the processed AnnData object.
adata_qc = sc.read_h5ad(OUTPUT_DIR / "filtered_scaled_umap_nodoublets_all_withall4plex.h5ad")


## 9. Harmony integration by sample

In [ ]:
# Make an independent copy of the selected data.
adata_qc_bysample = adata_qc.copy()
# Integrate using Harmony
sc.external.pp.harmony_integrate(
    adata_qc_bysample, 
    "sample"
)


In [ ]:
# Re-run neighbors analysis using the Harmony-adjusted PCA representation; no need for n_pcs, will be ignored if X_pca_harmony is used
#If want to change parameters then need to change them before integration
sc.pp.neighbors(adata_qc_bysample, use_rep='X_pca_harmony')

# Run UMAP using the neighbors computed from the Harmony-adjusted PCA
sc.tl.umap(adata_qc_bysample)

In [ ]:
# Visualize the UMAP results to assess integration 
sc.pl.umap(adata_qc_bysample, color='sample_type', title='After Harmony by Sample Integration UMAP Visualization' )

In [ ]:
# Visualize the UMAP results to assess integration 
sc.pl.umap(adata_qc_bysample, color='id', title='After Harmony by Sample Integration UMAP Visualization')

In [ ]:
# Visualize the UMAP results to assess integration 
sc.pl.umap(adata_qc_bysample, color='name', title='After Harmony by ID Integration UMAP Visualization')

In [ ]:
# Visualize the UMAP results to assess integration 
sc.pl.umap(adata_qc_bysample, color='plex', title='After Harmony by Sample Integration UMAP Visualization')

In [ ]:
# Visualize the UMAP results to assess integration 
sc.pl.umap(adata_qc_bysample, color='doublet_score', title='After Harmony by Sample Integration UMAP Visualization')

In [ ]:
# Save the processed AnnData object to disk.
adata_qc_bysample.write_h5ad(OUTPUT_DIR / "filtered_scaled_nodoublets_umap_Harmonybysample_all_withall4plex.h5ad")


In [ ]:
# Load the processed AnnData object.
adata_qc_bysample = sc.read_h5ad(OUTPUT_DIR / "filtered_scaled_nodoublets_umap_Harmonybysample_all_withall4plex.h5ad")


## 10. Leiden clustering and marker ranking

In [ ]:
# Using the igraph implementation and a fixed number of iterations can be significantly faster, especially for larger datasets
sc.tl.leiden(adata_qc_bysample, flavor="igraph", resolution=0.3)

In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = "leiden")


In [ ]:
# Identify genes associated with the selected clusters/groups.
sc.tl.rank_genes_groups(adata_qc_bysample, groupby = "leiden", method = "wilcoxon")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(adata_qc_bysample, groupby = "leiden", standard_scale="var", n_genes=10)


## 11. Manual cell-type annotation

Cluster-to-cell-type labels below are a biological interpretation step. For a public repository, document the marker evidence and rationale in the manuscript/README and update this mapping if clustering parameters change.

In [ ]:
# Map Leiden cluster IDs to cell type labels
cluster_map = {
    "0": "Pericytes",
    "1": "FBs",
    "2": "VCMs",
    "3": "Vasculature ECs",
    "4": "Myeloid",
    "5": "NCs",
    "6": "Lymphoid",
    "7": "SMCs",
    "8": "Endocardial ECs",
    "9": "LECs"
    # ... extend as needed
}

# Compute and store `adata_qc_bysample.obs['celltype']`.
adata_qc_bysample.obs["celltype"] = adata_qc_bysample.obs["leiden"].map(cluster_map)


In [ ]:
# Define the desired order of cell types
ordered_cell_types = ['VCMs', 'FBs', 'Pericytes', 'SMCs', 'Vasculature ECs', 'Endocardial ECs', 'LECs', 'Myeloid', 'Lymphoid', 'NCs']

# Reorder the 'cell_type' column in adata.obs
adata_qc_bysample.obs['celltype'] = pd.Categorical(adata_qc_bysample.obs['celltype'], categories=ordered_cell_types, ordered=True)

In [ ]:
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi = 300)
# Define the values used for `plt.rcParams['figure.figsize']`.
plt.rcParams['figure.figsize'] = [7,6]
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = "celltype",frameon=False)


In [ ]:
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi = 300)
# Define the values used for `plt.rcParams['figure.figsize']`.
plt.rcParams['figure.figsize'] = [7,6]
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = "sample_type",frameon=False)


### Cell-type UMAPs by experimental group

In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample[adata_qc_bysample.obs["sample_type"] == "R636Q"], color = "celltype", size = 3, title = "R636Q")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample[adata_qc_bysample.obs["sample_type"] == "BE"], color = "celltype", size = 3, title = "BE")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample[adata_qc_bysample.obs["sample_type"] == "WT"], color = "celltype", size = 3, title = "WT")


In [ ]:
# Inspect the number of observations in each category.
adata_qc_bysample.obs.sample_type.value_counts()


In [ ]:
# Inspect the number of observations in each category.
adata_qc_bysample.obs.id.value_counts()


In [ ]:
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi = 300)
# Compute and store `(fig, axes)`.
fig,axes=plt.subplots(1,3,figsize=(20,7))

# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample[adata_qc_bysample.obs["sample_type"] == "R636Q"], 
           color = "celltype", size = 3, ax=axes[0],title = "Mutant",legend_loc=None,show=False, frameon=False)
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample[adata_qc_bysample.obs["sample_type"] == "WT"], 
           color = "celltype", size = 8, ax=axes[1],title = "WT",legend_loc=None,show=False, frameon=False)
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample[adata_qc_bysample.obs["sample_type"] == "BE"], 
           color = "celltype", size = 8, ax=axes[2],title = "Base-edited",legend_loc=None,show=False, frameon=False)


# Run `plt.tight_layout` for this analysis step.
plt.tight_layout()
# Run `plt.show` for this analysis step.
plt.show()


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample[adata_qc_bysample.obs["sample_type"] == "BE"], 
           color = "celltype", size = 3,title = "WT",legend_loc=None,palette=None)


### Canonical marker visualization

In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = ["Pdgfra", "Col8a1"], frameon = False, legend_fontsize = 'large')


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = ["Ttn", "Tnnt2"], frameon = False, legend_fontsize = 'large')


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = ["Abcc9", "Pdgfrb"], frameon = False, legend_fontsize = 'large')


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = ["F13a1", "Mrc1"], frameon = False, legend_fontsize = 'large')


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = ["Ptprc", "Skap1"], frameon = False, legend_fontsize = 'large')


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = ["Myh11", "Acta2"], frameon = False, legend_fontsize = 'large')


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = ["Pecam1", "Flt1"], frameon = False, legend_fontsize = 'large')


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = ["Npr3", "Vwf"], frameon = False, legend_fontsize = 'large')


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = ["Reln", "Flt4"], frameon = False, legend_fontsize = 'large')


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(adata_qc_bysample, color = ["Zfp536", "Cadm2"], frameon = False, legend_fontsize = 'large')


In [ ]:
# List of your marker genes of interest
marker_genes = ["Tnnt2", "Col8a1", "Abcc9", "Myh11", "Pecam1", "Vwf", "Reln", "F13a1", "Skap1", "Cadm2"]
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi = 300)
# Run `sc.pl.stacked_violin` for this analysis step.
sc.pl.stacked_violin(
    adata_qc_bysample,
    marker_genes,
    groupby="celltype",
    swap_axes=False,      # optional: makes cell types on y-axis
    dendrogram=False,    # don’t cluster unless you want to
    figsize=(8,4),       # adjust as needed
    cmap="viridis"# colormap if continuous, but here we want by cell type
)


## 12. Cell-type composition summaries

In [ ]:
# Calculate the count of each cell type by sample
cell_type_by_sample = adata_qc_bysample.obs.groupby(['sample', 'celltype']).size().unstack(fill_value=0)

# Calculate the proportions of each cell type within each sample
cell_type_proportions_by_sample = cell_type_by_sample.divide(cell_type_by_sample.sum(axis=1), axis=0)

# View the proportions
print(cell_type_proportions_by_sample)

In [ ]:
# Plotting the stacked bar plot
plt.figure(figsize=(10, 8))

# Create a stacked bar plot
cell_type_proportions_by_sample.plot(kind='bar', stacked=True, figsize=(10, 6))

# Add labels and title
plt.xlabel('Sample')
# Run `plt.ylabel` for this analysis step.
plt.ylabel('Proportion')
# Run `plt.title` for this analysis step.
plt.title('Cell Type Proportions by Sample')
# Run `plt.xticks` for this analysis step.
plt.xticks(rotation=45, ha="right")
# Run `plt.legend` for this analysis step.
plt.legend(title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left')
# Run `plt.tight_layout` for this analysis step.
plt.tight_layout()  # Adjust layout to prevent clipping of labels

# Show the plot
plt.show()


In [ ]:
# Calculate the count of each cell type by sample
cell_type_by_sample = adata_qc_bysample.obs.groupby(['sample_type', 'celltype']).size().unstack(fill_value=0)

# Calculate the proportions of each cell type within each sample
cell_type_proportions_by_sample = cell_type_by_sample.divide(cell_type_by_sample.sum(axis=1), axis=0)

# View the proportions
print(cell_type_proportions_by_sample)

In [ ]:
# Plotting the stacked bar plot
mpl.rcParams['axes.grid'] = False

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(10, 2))
# Create a stacked bar plot
cell_type_proportions_by_sample.plot(kind='bar', stacked=True, figsize=(7, 6))

# Add labels and title
plt.xlabel('Sample_type')
# Run `plt.ylabel` for this analysis step.
plt.ylabel('Proportion')
# Run `plt.title` for this analysis step.
plt.title('Cell Type Proportions by Sample Type')
# Run `plt.xticks` for this analysis step.
plt.xticks(rotation=45, ha="right")
# Run `plt.legend` for this analysis step.
plt.legend(title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left')
# Run `plt.tight_layout` for this analysis step.
plt.tight_layout()  # Adjust layout to prevent clipping of labels

# Show the plot
plt.show()


In [ ]:
# Calculate the count of each cell type by sample
cell_type_by_plex = adata_qc_bysample.obs.groupby(['plex', 'celltype']).size().unstack(fill_value=0)

# Calculate the proportions of each cell type within each sample
cell_type_proportions_by_plex = cell_type_by_plex.divide(cell_type_by_plex.sum(axis=1), axis=0)

# View the proportions
print(cell_type_proportions_by_plex)

In [ ]:
# Plotting the stacked bar plot
plt.figure(figsize=(10, 8))

# Create a stacked bar plot
cell_type_proportions_by_plex.plot(kind='bar', stacked=True, figsize=(10, 6))

# Add labels and title
plt.xlabel('Plex')
# Run `plt.ylabel` for this analysis step.
plt.ylabel('Proportion')
# Run `plt.title` for this analysis step.
plt.title('Cell Type Proportions by Plex')
# Run `plt.xticks` for this analysis step.
plt.xticks(rotation=45, ha="right")
# Run `plt.legend` for this analysis step.
plt.legend(title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left')
# Run `plt.tight_layout` for this analysis step.
plt.tight_layout()  # Adjust layout to prevent clipping of labels

# Show the plot
plt.show()


## 13. Save annotated object and final QC plots

In [ ]:
# Save the processed AnnData object to disk.
adata_qc_bysample.write_h5ad(OUTPUT_DIR / "Allwithall4plex_filtered_integratedbysample_annotated.h5ad")


In [ ]:
# Load the processed AnnData object.
adata = sc.read_h5ad(OUTPUT_DIR / "Allwithall4plex_filtered_integratedbysample_annotated.h5ad")


In [ ]:
#Visualise
sc.set_figure_params(dpi = 300, figsize = (7,5))
# Plot the distribution of the selected measurement across groups.
sc.pl.violin(
    adata,
    ["n_genes_by_counts"],
    jitter=0, #default was 0.4 but gets too dense, cant see anything
    groupby="name"
)


In [ ]:
#Visualise
sc.pl.violin(
    adata,
    ["total_counts"],
    jitter=0, #default was 0.4 but gets too dense, cant see anything
    groupby="name"
)

In [ ]:
#Visualise
sc.pl.violin(
    adata,
    ["pct_counts_mt"],
    jitter=0, #default was 0.4 but gets too dense, cant see anything
    groupby="name"
)

In [ ]:
# Compute and store `value_counts`.
value_counts = adata.obs['name'].value_counts().sort_index()


In [ ]:
# Plot the bar chart
value_counts.plot(kind='bar')

# Add labels and title
plt.xlabel('Category')
# Run `plt.ylabel` for this analysis step.
plt.ylabel('Count')
# Run `plt.title` for this analysis step.
plt.title('Number of Filtered Cells')

# Show the plot
plt.show()
